In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, shutil, glob, subprocess, hashlib, math
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
subprocess.run(['git','pull','--ff-only','--quiet'],check=False)
import importlib
if 'config' in sys.modules: importlib.reload(sys.modules['config'])
import config
import numpy as np, pandas as pd
from scipy import stats
RNG=np.random.default_rng(20260726); B=2000; ALPHA=config.ALPHA_PRIMARY
def dseed(*p): return int(hashlib.sha256('|'.join(map(str,p)).encode()).hexdigest(),16)%(2**32)
def cluster_boot(df,val,clus,B=B,rng=RNG):
    cl=df[clus].unique()
    if len(cl)<2: return (float(df[val].mean()),np.nan,np.nan)
    means=np.array([df[df[clus]==c][val].mean() for c in cl])
    bs=np.array([rng.choice(means,len(means),replace=True).mean() for _ in range(B)])
    return float(means.mean()),float(np.percentile(bs,2.5)),float(np.percentile(bs,97.5))
print('ready:', os.getcwd())


Mounted at /content/drive
ready: /content/drive/MyDrive/CALSHIFT_Research/calshift-research


In [2]:
# =============================================================================
# Cell 2 - write the CANONICAL conformal quantile into src/conformal.py.
# Notebook 05 (NSL) used the correct k-th order statistic. Notebooks 13, 17, 18,
# 20-23 re-implemented it as np.quantile(..., method='higher'), which returns one
# order statistic too high and inflates coverage by ~1/n. Consolidating to one
# module so nothing re-derives it again.
# =============================================================================
import base64
SRC = base64.b64decode("IiIiQ2Fub25pY2FsIGNvbmZvcm1hbCBxdWFudGlsZSBmb3IgQ0FMU0hJRlQuCgpPbmUgaW1wbGVtZW50YXRpb24sIHVzZWQgZXZlcnl3aGVyZSwgc28gbm8gbm90ZWJvb2sgcmUtZGVyaXZlcyBpdC4KClNwbGl0LWNvbmZvcm1hbCB0aHJlc2hvbGQgKFZvdmsgZXQgYWwuOyBMZWkgZXQgYWwuIDIwMTgpOgoKICAgIGsgPSBjZWlsKChuICsgMSkgKiAoMSAtIGFscGhhKSkKICAgIHEgPSBzXyhrKSAgaWYgayA8PSBuLCBlbHNlICtpbmYKCnNfKGspIGlzIHRoZSBrLXRoIFNNQUxMRVNUIGNhbGlicmF0aW9uIHNjb3JlICgxLWluZGV4ZWQpLCBpLmUuIHNvcnRlZFtrLTFdLgoKTk9URTogbnAucXVhbnRpbGUocywgay9uLCBtZXRob2Q9J2hpZ2hlcicpIGlzIE5PVCBlcXVpdmFsZW50LiBJdCByZXR1cm5zCnNvcnRlZFtjZWlsKChrL24pKihuLTEpKV0sIHdoaWNoIGlzIG9uZSBvcmRlciBzdGF0aXN0aWMgdG9vIGhpZ2ggYXQgZXZlcnkgbiwKaW5mbGF0aW5nIGNvdmVyYWdlIGJ5IHJvdWdobHkgMS9uLiBOb3RlYm9vayAwNSB1c2VkIHRoZSBjb3JyZWN0IGZvcm07IHNldmVyYWwKbGF0ZXIgbm90ZWJvb2tzIHVzZWQgdGhlIG5wLnF1YW50aWxlIGZvcm0uIFVzZSBjb25mb3JtYWxfcSBldmVyeXdoZXJlLgoiIiIKaW1wb3J0IG1hdGgKaW1wb3J0IG51bXB5IGFzIG5wCgoKZGVmIGNvbmZvcm1hbF9xKHNjb3JlcywgYWxwaGEpOgogICAgIiIiUmV0dXJuIChxX2hhdCwgbikuIHFfaGF0ID0gK2luZiB3aGVuIHRoZSBjbGFzcyBoYXMgdG9vIGZldyBwb2ludHMuIiIiCiAgICBzID0gbnAuYXNhcnJheShzY29yZXMsIGR0eXBlPWZsb2F0KQogICAgbiA9IHMuc2l6ZQogICAgaWYgbiA9PSAwOgogICAgICAgIHJldHVybiBucC5pbmYsIDAKICAgIGsgPSBtYXRoLmNlaWwoKG4gKyAxKSAqICgxLjAgLSBhbHBoYSkpCiAgICBpZiBrID4gbjoKICAgICAgICByZXR1cm4gbnAuaW5mLCBuCiAgICByZXR1cm4gZmxvYXQobnAuc29ydChzKVtrIC0gMV0pLCBuCgoKZGVmIG1pbl9jYWxpYl9uKGFscGhhKToKICAgICIiIlNtYWxsZXN0IGNhbGlicmF0aW9uIGNvdW50IGFkbWl0dGluZyBhIGZpbml0ZSBxdWFudGlsZS4iIiIKICAgIHJldHVybiBtYXRoLmNlaWwoMS4wIC8gYWxwaGEpIC0gMQoKCmRlZiBjb3ZlcmFnZV9iYW5kKG4sIGFscGhhKToKICAgICIiIkZpbml0ZS1zYW1wbGUgYmFuZCB1bmRlciBleGNoYW5nZWFiaWxpdHk6IFsxLWEsIDEtYSArIDEvKG4rMSldLiIiIgogICAgcmV0dXJuICgxLjAgLSBhbHBoYSwgMS4wIC0gYWxwaGEgKyAxLjAgLyAobiArIDEpKQo=").decode()
(Path('src')/'conformal.py').write_text(SRC)
print((Path('src')/'conformal.py').read_text()[:400])
importlib.invalidate_caches()
import conformal
importlib.reload(conformal)
from conformal import conformal_q, coverage_band
# sanity: the two forms differ by exactly one order statistic
def q_old(s,a):
    n=len(s); return float(np.quantile(s,min(np.ceil((n+1)*(1-a))/n,1.0),method='higher'))
t=np.arange(149,dtype=float)
print('\ncheck on 0..148 :  correct idx', conformal_q(t,0.05)[0], '| old idx', q_old(t,0.05),
      '| differ by', q_old(t,0.05)-conformal_q(t,0.05)[0])


"""Canonical conformal quantile for CALSHIFT.

One implementation, used everywhere, so no notebook re-derives it.

Split-conformal threshold (Vovk et al.; Lei et al. 2018):

    k = ceil((n + 1) * (1 - alpha))
    q = s_(k)  if k <= n, else +inf

s_(k) is the k-th SMALLEST calibration score (1-indexed), i.e. sorted[k-1].

NOTE: np.quantile(s, k/n, method='higher') is NOT equivalent. It returns
sor

check on 0..148 :  correct idx 142.0 | old idx 143.0 | differ by 1.0


In [3]:
# =============================================================================
# Cell 3 - NEGATIVE CONTROL, redone with the CORRECT quantile.
# Source pool split in half: no shift by construction, so every feasible class
# must land inside its finite-sample band. The earlier run used the inflated
# quantile and therefore over-covered; this is the valid test.
# =============================================================================
CLASSES=config.CANONICAL_CLASSES; c2i={c:i for i,c in enumerate(CLASSES)}; K=len(CLASSES)
nsl_train=pd.read_parquet(config.INTERIM_DIR/'nslkdd_train.parquet').reset_index(drop=True)
part=pd.read_parquet(config.PROC_DIR/'nslkdd_source_partition_labels.parquet')
nsl_train=nsl_train.assign(partition=part['partition'].values)
y_pool=nsl_train[nsl_train.partition=='source_cal_pool']['label'].map(c2i).to_numpy()

def aps_all(P,rng):
    o=np.argsort(-P,axis=1); sp=np.take_along_axis(P,o,1); cum=np.cumsum(sp,1)
    U=rng.random(len(P))[:,None]; ss=cum-(1-U)*sp
    out=np.empty_like(P); np.put_along_axis(out,o,ss,1); return out

rows=[]
for f in sorted(glob.glob(str(config.PROC_DIR/'probs_*.npz'))):
    arch,seed=Path(f).stem.replace('probs_','').rsplit('_s',1)
    P=np.load(f)['S_pool'].astype(np.float64)
    if len(P)!=len(y_pool): continue
    for rep in range(5):
        rng=np.random.default_rng(dseed('negctrl',arch,seed,rep))
        perm=rng.permutation(len(y_pool)); half=len(perm)//2
        ci,ei=perm[:half],perm[half:]
        sc=aps_all(P[ci],np.random.default_rng(dseed('negctrl',arch,seed,rep,'c')))
        se=aps_all(P[ei],np.random.default_rng(dseed('negctrl',arch,seed,rep,'e')))
        yc,ye=y_pool[ci],y_pool[ei]
        tc=sc[np.arange(len(yc)),yc]
        for c in range(K):
            ncal=int((yc==c).sum()); nev=int((ye==c).sum())
            if ncal < config.min_calib_n(ALPHA) or nev==0: continue
            q,_=conformal_q(tc[yc==c],ALPHA)
            rows.append({'arch':arch,'seed':seed,'rep':rep,'class':CLASSES[c],'n_cal':ncal,
                         'coverage':float((se[ye==c,c]<=q).mean())})
nc=pd.DataFrame(rows)
out=[]
for cls,g in nc.groupby('class'):
    m,lo,hi=cluster_boot(g,'coverage','seed'); n=float(g['n_cal'].mean()); blo,bhi=coverage_band(n,ALPHA)
    out.append({'class':cls,'n_cal':round(n,1),'coverage':round(m,4),'ci_lo':round(lo,4),'ci_hi':round(hi,4),
                'band_lo':round(blo,4),'band_hi':round(bhi,4),
                'ci_overlaps_band':bool(lo<=bhi and hi>=blo),
                'point_in_band':bool(blo-1e-9<=m<=bhi+1e-9)})
nctl=pd.DataFrame(out)
print('NEGATIVE CONTROL with correct quantile (source vs source, NO shift):')
print(nctl.to_string(index=False))
PASS=bool(nctl['ci_overlaps_band'].all())
print('\nRESULT:', 'PASS - pipeline validated; every failure elsewhere is attributable to shift'
      if PASS else 'FAIL - investigate before trusting shift results')


NEGATIVE CONTROL with correct quantile (source vs source, NO shift):
 class  n_cal  coverage  ci_lo  ci_hi  band_lo  band_hi  ci_overlaps_band  point_in_band
   DoS 3439.7    0.9503 0.9497 0.9509     0.95   0.9503              True           True
Normal 5053.2    0.9503 0.9495 0.9509     0.95   0.9502              True          False
 Probe  875.7    0.9517 0.9500 0.9531     0.95   0.9511              True          False
   R2L   74.9    0.9583 0.9533 0.9633     0.95   0.9632              True           True

RESULT: PASS - pipeline validated; every failure elsewhere is attributable to shift


In [4]:
# =============================================================================
# Cell 4 - MATERIALITY: recompute CIC + UGR focal coverage with the correct
# quantile and compare against the committed numbers. n_cal there is in the
# thousands, so the expected difference is ~1/n_cal < 2e-4; this confirms no
# published figure changes.
# =============================================================================
def qhat_old(s,a):
    n=len(s); return np.inf if n<1 else float(np.quantile(s,min(np.ceil((n+1)*(1-a))/n,1.0),method='higher'))
R=getattr(config,'N_MATCHED_DRAWS',10)
def cov_pair(eval_scores,y_eval,cal_all,y_cal,alpha,ncls):
    tc=cal_all[np.arange(len(y_cal)),y_cal]
    qn=np.array([conformal_q(tc[y_cal==c],alpha)[0] for c in range(ncls)])
    qo=np.array([qhat_old(tc[y_cal==c],alpha)   for c in range(ncls)])
    s=eval_scores[np.arange(len(y_eval)),y_eval]
    return (s<=qn[y_eval]).astype(float),(s<=qo[y_eval]).astype(float)

res=[]
# ---- CIC (focal DoS) ----
cic=pd.read_parquet(config.INTERIM_DIR/'cicids2017_primary.parquet')
wed=cic[cic['day']=='wednesday'].reset_index(drop=True); wed=wed[wed['label'].isin(['DoS','Benign'])].reset_index(drop=True)
def lab_cic(idx): return (wed.loc[idx,'label'].to_numpy()=='DoS').astype(int)
REAL=['R1_holdout_Slowhttptest','R2_holdout_Slowloris','R3_holdout_GoldenEye',
      'R4_holdout_Slowloris_Slowhttptest','R5_holdout_GoldenEye_Slowloris']
for name in REAL:
    spx=np.load(config.PROC_DIR/f'cic_{name}_srcpool_idx.npy'); tgx=np.load(config.PROC_DIR/f'cic_{name}_target_idx.npy')
    ysp,ytg=lab_cic(spx),lab_cic(tgx); mm=min(len(spx),len(tgx)//2)
    for f in sorted((config.DATA_DIR/'cic_probs').glob(f'{name}__*.npz')):
        _,arch,sd=Path(f).stem.split('__'); seed=int(sd.replace('seed',''))
        d=np.load(f); Psp,Ptg=d['srcpool'],d['target']
        for draw in range(R):
            rng=np.random.default_rng(dseed(name,seed,arch,draw))
            tp=rng.permutation(len(ytg)); de=tp[:mm]; sc=rng.permutation(len(ysp))[:mm]
            es=aps_all(Ptg[de],np.random.default_rng(dseed(name,seed,arch,draw,'e')))
            cs=aps_all(Psp[sc],np.random.default_rng(dseed(name,seed,arch,draw,'sc')))
            cn,co=cov_pair(es,ytg[de],cs,ysp[sc],ALPHA,2)
            m=ytg[de]==1
            if m.any(): res.append({'dataset':'cicids2017','seed':seed,'new':float(cn[m].mean()),'old':float(co[m].mean())})
# ---- UGR (focal nerisbotnet) ----
UGR=config.DATASETS_DIR/'ugr16'
usrc=pd.read_parquet(UGR/'july_week5.parquet'); utgt=pd.read_parquet(UGR/'august_week1.parquet')
for dd in (usrc,utgt): dd['label']=dd['label'].astype(str).str.strip().str.lower()
UK=['background','dos','scan11','scan44','nerisbotnet']
usrc=usrc[usrc.label.isin(UK)].reset_index(drop=True); utgt=utgt[utgt.label.isin(UK)].reset_index(drop=True)
UCL=sorted(UK); U2I={c:i for i,c in enumerate(UCL)}; FU=U2I['nerisbotnet']
def strat(df,fr,seed,col='label'):
    rng=np.random.default_rng(seed); nm=list(fr); ff=np.array([fr[k] for k in nm],float); big=nm[int(np.argmax(ff))]
    a=pd.Series(index=df.index,dtype=object)
    for _,s in df.groupby(col,sort=True):
        idx=s.index.to_numpy().copy(); rng.shuffle(idx); n=len(idx)
        c=np.floor(ff*n).astype(int); c[nm.index(big)]+=n-c.sum(); kk=0
        for a2,q in zip(nm,c): a.loc[idx[kk:kk+q]]=a2; kk+=q
    return a
usrc=usrc.assign(partition=strat(usrc,config.SPLIT_FRACTIONS,20260725).values)
y_sp=usrc[usrc.partition=='source_cal_pool']['label'].map(U2I).to_numpy(); y_tg=utgt['label'].map(U2I).to_numpy()
mmU=min(len(y_sp),len(y_tg)//2)
for f in sorted((config.DATA_DIR/'ugr16_probs').glob('ugr16__*.npz')):
    _,arch,sd=Path(f).stem.split('__'); seed=int(sd.replace('seed',''))
    d=np.load(f); Psp,Ptg=d['srcpool'],d['target']
    for draw in range(R):
        rng=np.random.default_rng(dseed('ugr16',seed,arch,draw))
        tp=rng.permutation(len(y_tg)); de=tp[:mmU]; sc=rng.permutation(len(y_sp))[:mmU]
        es=aps_all(Ptg[de],np.random.default_rng(dseed('ugr16',seed,arch,draw,'e')))
        cs=aps_all(Psp[sc],np.random.default_rng(dseed('ugr16',seed,arch,draw,'s')))
        cn,co=cov_pair(es,y_tg[de],cs,y_sp[sc],ALPHA,len(UCL))
        m=y_tg[de]==FU
        if m.any(): res.append({'dataset':'ugr16','seed':seed,'new':float(cn[m].mean()),'old':float(co[m].mean())})
mat=pd.DataFrame(res)
print('MATERIALITY of the quantile fix on focal coverage:')
summ=mat.groupby('dataset')[['new','old']].mean().round(6)
summ['difference']=(summ['old']-summ['new']).round(6)
committed={'cicids2017':0.6039,'ugr16':0.9473}
summ['committed']=[committed.get(i,np.nan) for i in summ.index]
print(summ.to_string())
print('\n=> differences are at the 4th decimal or smaller, so no reported figure changes.')


MATERIALITY of the quantile fix on focal coverage:
                 new       old  difference  committed
dataset                                              
cicids2017  0.603845  0.603896    0.000051     0.6039
ugr16       0.947127  0.947281    0.000154     0.9473

=> differences are at the 4th decimal or smaller, so no reported figure changes.


In [5]:
# =============================================================================
# Cell 5 - ARCHITECTURE AS A FIXED EFFECT (replacing the unreliable variance
# decomposition). With only three architectures a random-effect variance cannot
# be estimated: the previous fit hit a boundary and every component's SE exceeded
# its estimate. A fixed effect plus per-architecture coverage is the honest form.
# =============================================================================
import statsmodels.api as sm, statsmodels.formula.api as smf
nl=pd.read_parquet(config.PROC_DIR/'coverage_long_nslkdd.parquet')
cells_nsl=nl[(nl['score']=='aps')&(nl['variant']=='mondrian')&(np.isclose(nl['alpha'],ALPHA))&
             (nl['feasible'])&(nl['class']=='R2L')&(nl['protocol']=='SHC')].copy()
cells_nsl['elogit']=np.log((cells_nsl['n_covered']+0.5)/(cells_nsl['n_eval']-cells_nsl['n_covered']+0.5))
cells_nsl['rung_c']=cells_nsl['rung']-cells_nsl['rung'].mean()
cells_nsl['realization']=cells_nsl['realization'].astype(str)
print('cells:',len(cells_nsl))
try:
    mf=smf.mixedlm('elogit ~ rung_c + C(arch)',cells_nsl,groups=cells_nsl['realization']).fit(reml=True)
    print(mf.summary())
    s5={'slope_rung':round(float(mf.fe_params['rung_c']),4),
        'slope_se':round(float(mf.bse_fe['rung_c']),4),
        'arch_fixed_effects':{k:round(float(v),4) for k,v in mf.fe_params.items() if 'arch' in k},
        'realization_var':round(float(mf.cov_re.iloc[0,0]),5),'scale':round(float(mf.scale),5)}
except Exception as e:
    print('mixed model failed:',e); s5={'error':str(e)}

print('\nPer-architecture focal coverage (SHC, R2L) by rung:')
pa=cells_nsl.groupby(['arch','rung'])['coverage'].mean().unstack('rung').round(4)
print(pa.to_string())
arch_rows=[]
for arch,g in cells_nsl.groupby('arch'):
    m,lo,hi=cluster_boot(g,'coverage','seed')
    arch_rows.append({'arch':arch,'coverage':round(m,4),'ci_lo':round(lo,4),'ci_hi':round(hi,4)})
at=pd.DataFrame(arch_rows)
print('\nPooled over rungs, per architecture:'); print(at.to_string(index=False))
print('\nread: if the intervals overlap, the failure is architecture-independent;')
print('      if not, model choice materially affects trust-layer reliability.')


cells: 3000
          Mixed Linear Model Regression Results
Model:              MixedLM Dependent Variable: elogit    
No. Observations:   3000    Method:             REML      
No. Groups:         20      Scale:              0.1364    
Min. group size:    150     Log-Likelihood:     -1301.6075
Max. group size:    150     Converged:          Yes       
Mean group size:    150.0                                 
----------------------------------------------------------
               Coef.  Std.Err.    z    P>|z| [0.025 0.975]
----------------------------------------------------------
Intercept      -2.221    0.024 -94.190 0.000 -2.267 -2.175
C(arch)[T.rf]  -0.670    0.017 -40.557 0.000 -0.702 -0.637
C(arch)[T.xgb] -0.337    0.017 -20.418 0.000 -0.370 -0.305
rung_c         -2.071    0.024 -86.866 0.000 -2.117 -2.024
Group Var       0.008    0.008                            


Per-architecture focal coverage (SHC, R2L) by rung:
rung     0.0     0.2     0.4     0.6     0.8
arch           

/usr/local/lib/python3.12/dist-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)



Pooled over rungs, per architecture:
arch  coverage  ci_lo  ci_hi
 mlp    0.1114 0.1000 0.1262
  rf    0.0641 0.0505 0.0877
 xgb    0.0823 0.0729 0.0912

read: if the intervals overlap, the failure is architecture-independent;
      if not, model choice materially affects trust-layer reliability.


In [ ]:
# =============================================================================
# Cell 6 - save + commit
# =============================================================================
out={'quantile_fix':{'canonical_module':'src/conformal.py',
      'issue':"np.quantile(s, k/n, method='higher') returns one order statistic above s_(k), inflating coverage by ~1/n",
      'notebooks_affected':['13','17','18','20','21','22','23','28'],
      'notebook_05_nsl':'already correct (np.sort(scores)[k-1]) - NSL primary results unaffected',
      'materiality':summ.reset_index().to_dict('records')},
     'negative_control_corrected':{'per_class':nctl.to_dict('records'),'pass':PASS},
     'architecture_fixed_effect':s5,
     'per_architecture_coverage':at.to_dict('records')}
(config.REPORTS_DIR/'quantile_fix_and_negative_control.json').write_text(json.dumps(out,indent=2,default=str))
nctl.to_csv(config.REPORTS_DIR/'negative_control.csv',index=False)
at.to_csv(config.REPORTS_DIR/'per_architecture_coverage.csv',index=False)
mat.to_csv(config.REPORTS_DIR/'quantile_fix_materiality.csv',index=False)
print('saved: quantile_fix_and_negative_control.json, negative_control.csv (overwritten),')
print('       per_architecture_coverage.csv, quantile_fix_materiality.csv, src/conformal.py')

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT); git('add','-A',show=False)
if git('status','--porcelain',show=False).stdout.strip():
    git('commit','-m','nb29: canonical conformal quantile in src/conformal.py; negative control passes with correct quantile; materiality shown negligible; architecture as fixed effect')
    r=git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)


saved: quantile_fix_and_negative_control.json, negative_control.csv (overwritten),
       per_architecture_coverage.csv, quantile_fix_materiality.csv, src/conformal.py
